# Modell 3 Leave-One-Driver-Out

### Was kann dieses Modell?

### Daten: Ein Rennen (Belgien)
### Modelle: LogReg, XGBoost
### Training: 19/1 Fahrer
### Vorhersage: Probs für ein Fahrer
### Problem: ???

## Imports

In [1]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

## Rennen definieren

In [2]:
YEAR = 2024
GRAND_PRIX = ["Belgian Grand Prix"]
SESSION_TYPE = "R"

## Daten laden

In [3]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

req         WARNING 	DEFAULT CACHE ENABLED! (2.5 GB) /Users/lukaskoch/Library/Caches/fastf1
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	

✔ Loaded Belgian Grand Prix


## Einheitliche Features laden

In [4]:
fe = FeatureEngineer()
merged = fe.create_features(data)

## Features auswählen

In [5]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_MEDIUM',
 'compound_SOFT',
 'status_1',
 'status_12']

In [6]:
models = Models.define_models()

In [7]:
from utils.experiment_db import log_or_update_metrics
from utils.Metrics import Metrics

drivers = merged["Driver"].unique()

def train_and_evaluate(model, model_name):
    """
    Leave-One-Driver-Out Cross-Validation mit Metrics-Utility
    """

    fold_metrics = []      # speichert pro Fahrer-Metrics
    per_driver_rows = []   # für Ausgabe-DataFrame

    for driver in drivers:
        print(f"\n🚗 Leave-One-Driver-Out: Testing on {driver}")

        # Split pro Fahrer
        train_df = merged[merged["Driver"] != driver]
        test_df  = merged[merged["Driver"] == driver]

        X_train = train_df[features]
        y_train = train_df["pit_in_next_lap"]

        X_test = test_df[features]
        y_test = test_df["pit_in_next_lap"]

        # Model
        model.fit(X_train, y_train)

        # Predictions
        y_proba = model.predict_proba(X_test)[:, 1]

        # 👉 Nutzung der Metrics.calculate_metrics()
        metrics = Metrics.calculate_metrics(
            model_name=driver,
            y_test=y_test,
            y_pred_prob=y_proba,
        )

        fold_metrics.append(metrics)

        # Für das späteres DF
        per_driver_rows.append({
            "driver": driver,
            **metrics
        })

    # 👉 Aggregation über ALLE driver-folds
    avg_metrics = Metrics.aggregate_cv_metrics(
        list_of_fold_metrics=fold_metrics,
        model_name= f"approach_03_{model_name}"
    )

    # Ausgabe DF
    results_df = pd.DataFrame(per_driver_rows)
    avg_row = {"driver": "AVERAGE", **avg_metrics}
    results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)
    results_df = results_df.drop(columns=["model_name"])

    

    # 👉 Logging
    log_or_update_metrics(
        model_name=f"approach_03_{model_name}",
        metrics={
            "model_name": f"approach_03_{model_name}",
            **avg_metrics
        }
    )
    print(results_df)

    return results_df


In [8]:
scores = {}
for name, model in models.items():
    metrics = train_and_evaluate(model, name)
    scores[name] = metrics
    


🚗 Leave-One-Driver-Out: Testing on LEC

🚗 Leave-One-Driver-Out: Testing on HAM

🚗 Leave-One-Driver-Out: Testing on PER

🚗 Leave-One-Driver-Out: Testing on PIA

🚗 Leave-One-Driver-Out: Testing on RUS

🚗 Leave-One-Driver-Out: Testing on SAI

🚗 Leave-One-Driver-Out: Testing on NOR

🚗 Leave-One-Driver-Out: Testing on ALO

🚗 Leave-One-Driver-Out: Testing on VER

🚗 Leave-One-Driver-Out: Testing on ALB

🚗 Leave-One-Driver-Out: Testing on OCO

🚗 Leave-One-Driver-Out: Testing on GAS

🚗 Leave-One-Driver-Out: Testing on BOT

🚗 Leave-One-Driver-Out: Testing on RIC

🚗 Leave-One-Driver-Out: Testing on STR

🚗 Leave-One-Driver-Out: Testing on HUL

🚗 Leave-One-Driver-Out: Testing on MAG

🚗 Leave-One-Driver-Out: Testing on ZHO

🚗 Leave-One-Driver-Out: Testing on TSU

🚗 Leave-One-Driver-Out: Testing on SAR
     driver  brier_score  log_loss   roc_auc    pr_auc
0       LEC     0.188693  0.560704  0.714286  0.538462
1       HAM     0.187340  0.559506  0.630952  0.158333
2       PER     0.215699  0.615787 